# Notebook 04 — Analyse sémantique (LDA, Word2Vec, CamemBERT)

## Objectif
Ce notebook explore la **structure sémantique** des discours sur les VSS grâce à trois méthodes complémentaires :

1. **LDA (Latent Dirichlet Allocation)** : détecte automatiquement les thématiques principales dans les discours. Permet d'identifier si un "topic identitaire" émerge et son poids relatif par bloc.

2. **Word2Vec / CamemBERT** : transforme chaque discours en un vecteur numérique. On mesure ensuite la **similarité cosinus** entre les blocs pour voir si leurs discours convergent.

3. **Concept ciblé** : on crée un vecteur "immigration/identité" et on mesure comment chaque bloc s'en rapproche au fil du temps.

## Hypothèse testée
> *L'extrême droite cadre de plus en plus les VSS dans un vocabulaire identitaire/migratoire, et les autres blocs (en particulier la droite traditionnelle) convergent vers ce cadrage.*

## Entrées
- `df_vss_propre.pkl` (notebook 02)

## Sorties
- Graphiques de convergence sémantique
- `df_vss_embeddings.pkl` (avec les vecteurs CamemBERT pré-calculés)

## 1. Imports et chargement

In [ ]:
from config import *
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re, os
from tqdm import tqdm

from gensim import corpora, models
from gensim.models import Word2Vec
from sklearn.metrics.pairwise import cosine_similarity

import nltk
from nltk.stem.snowball import FrenchStemmer
nltk.download('stopwords', quiet=True)

stemmer = FrenchStemmer()

# Chargement
df_vss = pd.read_pickle(CHEMIN_DF_VSS_PROPRE)
df_vss['date'] = pd.to_datetime(df_vss['date'])
print(f"✅ {len(df_vss)} prises de parole chargées.")

## 2. Topic Modeling (LDA)

### Principe
L'algorithme LDA découvre automatiquement des "topics" (thématiques) à partir de la co-occurrence des mots dans les documents. Chaque document est un mélange de topics, et chaque topic est un mélange de mots.

### Pourquoi retirer les mots-clés VSS ?
Pour éviter un biais circulaire : puisqu'on a filtré les documents *par* ces mots-clés, ils apparaîtraient dans tous les topics. On les retire pour voir ce qui *accompagne* les discours VSS.

In [ ]:
# Préparation du corpus LDA
df_blocs = df_vss.dropna(subset=['bloc', 'texte_clean']).copy()
df_blocs = df_blocs[df_blocs['texte_clean'].str.strip() != ""]

textes = [str(doc).split() for doc in df_blocs['texte_clean']]

dictionnaire = corpora.Dictionary(textes)
dictionnaire.filter_extremes(no_below=5, no_above=0.4)

corpus = [dictionnaire.doc2bow(text) for text in textes]

print(f"✅ Corpus préparé : {len(corpus)} documents, {len(dictionnaire)} termes uniques.")

In [ ]:
# Entraînement du modèle LDA
NUM_TOPICS = 15

lda = models.LdaModel(
    corpus=corpus,
    id2word=dictionnaire,
    num_topics=NUM_TOPICS,
    passes=20,
    random_state=42
)

print(f"\n📊 {NUM_TOPICS} thématiques découvertes :\n")
for i, topic in lda.print_topics(-1):
    print(f"  Topic #{i}: {topic}")

### Évolution d'un topic par bloc

Après avoir identifié visuellement le topic qui correspond le mieux au "cadrage identitaire/immigration" (ex : topic #4 ou #13), on trace son poids moyen par bloc et par année.

> **⚠️ Ajustez `TOPIC_IDENTITAIRE`** en fonction des résultats LDA ci-dessus.

In [ ]:
# --- PARAMÈTRE À AJUSTER ---
TOPIC_IDENTITAIRE = 4  # Numéro du topic à suivre (à ajuster après inspection)

# Extraction des poids
poids = []
for bow in corpus:
    dist = dict(lda.get_document_topics(bow))
    poids.append(dist.get(TOPIC_IDENTITAIRE, 0))

df_blocs_lda = df_blocs.copy()
df_blocs_lda['poids_topic'] = poids
df_blocs_lda['annee'] = df_blocs_lda['date'].dt.to_period('Y').astype(str)

df_evol = df_blocs_lda.groupby(['annee', 'bloc'])['poids_topic'].mean().reset_index()

plt.figure(figsize=(12, 6))
sns.lineplot(data=df_evol, x='annee', y='poids_topic',
             hue='bloc', hue_order=ORDRE_BLOCS, marker='o', linewidth=2.5)
plt.title(f"Poids du topic #{TOPIC_IDENTITAIRE} dans les discours VSS par bloc")
plt.ylabel("Probabilité moyenne du topic")
plt.xlabel("Année")
plt.legend(title="Bloc", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

## 3. Similarité cosinus avec CamemBERT

### Principe
Au lieu d'utiliser Word2Vec (qui ne capte pas bien le sens contextuel), on utilise **CamemBERT** — un modèle de langue pré-entraîné sur du français. Chaque discours est transformé en un vecteur de 768 dimensions.

On calcule ensuite le **centroïde** (vecteur moyen) de chaque bloc pour chaque année, et on mesure la similarité cosinus entre chaque bloc et l'extrême droite.

> **⏱ Temps estimé** : ~30-60 min pour encoder les ~10 000 discours.
> **Nécessite** : `pip install torch transformers`

In [ ]:
# Installation de torch et transformers si nécessaire
import subprocess, sys

try:
    import torch
    from transformers import CamembertTokenizer, CamembertModel
    TORCH_OK = True
except ImportError:
    print("⏳ Installation de torch et transformers...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch", "transformers", "sentencepiece"])
    import torch
    from transformers import CamembertTokenizer, CamembertModel
    TORCH_OK = True

print(f"✅ PyTorch et Transformers disponibles.")
print(f"   GPU disponible : {torch.cuda.is_available()}")

In [ ]:
# ==========================================================================
# CACHE : Si les embeddings existent déjà, on les charge
# ==========================================================================

if os.path.exists(CHEMIN_DF_EMBEDDINGS):
    df_blocs = pd.read_pickle(CHEMIN_DF_EMBEDDINGS)
    print(f"⏩ Embeddings chargés depuis le cache : {len(df_blocs)} prises de parole.")
else:
    print("🔄 Encodage CamemBERT des prises de parole...")

    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = CamembertTokenizer.from_pretrained("camembert-base")
    model = CamembertModel.from_pretrained("camembert-base").to(device)
    model.eval()

    def encode_texte(texte):
        inputs = tokenizer(
            str(texte), return_tensors="pt",
            truncation=True, padding=True, max_length=512
        ).to(device)
        with torch.no_grad():
            outputs = model(**inputs)
        return outputs.last_hidden_state[:, 0, :].squeeze().cpu().numpy()

    df_blocs = df_vss.dropna(subset=['bloc']).copy()

    tqdm.pandas(desc="Encodage CamemBERT")
    df_blocs['vecteur'] = df_blocs['texte'].progress_apply(encode_texte)

    df_blocs.to_pickle(CHEMIN_DF_EMBEDDINGS)
    print(f"💾 Embeddings sauvegardés dans {CHEMIN_DF_EMBEDDINGS}")

### Similarité cosinus entre blocs et extrême droite

In [ ]:
df_blocs['annee'] = df_blocs['date'].dt.to_period('Y').astype(str)

resultats_sim = []
for annee in sorted(df_blocs['annee'].unique()):
    df_an = df_blocs[df_blocs['annee'] == annee]

    if "Extrême Droite" not in df_an['bloc'].values:
        continue

    vecs_ed = np.vstack(df_an[df_an['bloc'] == "Extrême Droite"]['vecteur'].values)
    centroid_ed = np.mean(vecs_ed, axis=0)

    for bloc in df_an['bloc'].unique():
        if bloc == "Extrême Droite":
            continue
        vecs = np.vstack(df_an[df_an['bloc'] == bloc]['vecteur'].values)
        centroid = np.mean(vecs, axis=0)
        sim = cosine_similarity([centroid_ed], [centroid])[0][0]
        resultats_sim.append({'annee': annee, 'bloc': bloc, 'similarite': sim})

df_sim = pd.DataFrame(resultats_sim)

palette = {
    "Droite Traditionnelle": "#1E3A8A", "Centre": "#D97706",
    "Gauche Modérée": "#166534", "Gauche Radicale": "#DC2626"
}

plt.figure(figsize=(12, 6))
sns.lineplot(data=df_sim, x='annee', y='similarite', hue='bloc',
             palette=palette, marker='o', linewidth=2.5)
plt.title("Proximité sémantique des blocs avec l'extrême droite\n(CamemBERT, similarité cosinus)")
plt.ylabel("Similarité cosinus")
plt.xlabel("Année")
plt.legend(title="Bloc", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, linestyle='--', alpha=0.5)
plt.ylim(df_sim['similarite'].min() - 0.02, 1.0)
plt.tight_layout()
plt.show()

## 4. Convergence vers le concept "immigration/identité"

Plutôt que de comparer les blocs entre eux, on crée un **vecteur de référence** représentant le concept "immigration/identité" et on mesure comment chaque bloc s'en rapproche.

In [ ]:
# Création du vecteur concept
mots_concept = [
    "immigration", "clandestin", "frontière", "national", "identité",
    "culture", "étranger", "expulsion", "valeurs", "OQTF", "territoire"
]

# On encode la phrase-concept avec CamemBERT
if 'encode_texte' not in dir():
    # Si on a chargé depuis le cache, on a besoin de recréer la fonction
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = CamembertTokenizer.from_pretrained("camembert-base")
    model_camembert = CamembertModel.from_pretrained("camembert-base").to(device)
    model_camembert.eval()
    def encode_texte(texte):
        inputs = tokenizer(str(texte), return_tensors="pt", truncation=True, padding=True, max_length=512).to(device)
        with torch.no_grad():
            outputs = model_camembert(**inputs)
        return outputs.last_hidden_state[:, 0, :].squeeze().cpu().numpy()

vecteur_concept = encode_texte(" ".join(mots_concept))

# Mesure de proximité pour chaque discours
df_blocs['proximite_concept'] = df_blocs['vecteur'].apply(
    lambda x: cosine_similarity([x], [vecteur_concept])[0][0]
)

# Moyenne par bloc et par an
df_derive = df_blocs.groupby(['annee', 'bloc'])['proximite_concept'].mean().reset_index()

couleurs = {
    "Extrême Droite": "#8B0000", "Droite Traditionnelle": "#0000FF",
    "Centre": "#FFA500", "Gauche Modérée": "#228B22", "Gauche Radicale": "#FF0000"
}

plt.figure(figsize=(12, 6))
sns.lineplot(data=df_derive, x='annee', y='proximite_concept',
             hue='bloc', hue_order=ORDRE_BLOCS,
             palette=couleurs, marker='o', linewidth=3, markersize=8)
plt.title("Convergence vers le cadrage 'immigration/identité' dans les discours VSS")
plt.ylabel("Similarité avec le concept identitaire (cosinus)")
plt.xlabel("Année")
plt.legend(title="Bloc", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()